In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [64]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score

In [65]:
df=pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [66]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [67]:
df=df.dropna()

In [68]:
df.drop(columns=['customerID'],axis=1,inplace=True)


In [69]:
X=df.drop(columns=['Churn'],axis=1)
y=df['Churn']

In [70]:
num_faeture=X.select_dtypes(exclude='object').columns
cat_feature=X.select_dtypes(include="object").columns

In [71]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

numerical_feature=StandardScaler()
categorical_feature=OneHotEncoder(drop='first')


preprocessor=ColumnTransformer(
    [
        ('onehotencoder',categorical_feature,cat_feature),
        ('StandardScaler',numerical_feature,num_faeture)
    ],remainder='passthrough'
)

In [72]:
X=preprocessor.fit_transform(X)

In [73]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440327,-1.280248,-1.161694,-0.994194
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,-0.440327,0.064303,-0.260878,-0.173740
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,-0.440327,-1.239504,-0.363923,-0.959649
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,-0.440327,0.512486,-0.747850,-0.195248
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440327,-1.239504,0.196178,-0.940457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,-0.440327,-0.343137,0.664868,-0.129180
7028,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,-0.440327,1.612573,1.276493,2.241056
7029,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,-0.440327,-0.872808,-1.170004,-0.854514
7030,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,2.271039,-1.158016,0.319168,-0.872095


In [74]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

In [75]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

Churn
No     3872
Yes    1402
Name: count, dtype: int64
Churn
No     0.734168
Yes    0.265832
Name: proportion, dtype: float64


In [76]:
def evaluate(true,pred):
    AccuracyScore=accuracy_score(true,pred)
    f1Score=f1_score(true,pred,pos_label='Yes')
    precisionScore=precision_score(true,pred,pos_label='Yes')
    recallScore=recall_score(true,pred,pos_label='Yes')
    
    return AccuracyScore,f1Score,precisionScore,recallScore

In [77]:
models={
    "RandomForestClassifier":RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42),
    "GradientBoostingClassifier":GradientBoostingClassifier(random_state=42),
    "DecisionTreeClassifier":DecisionTreeClassifier(class_weight='balanced',random_state=42),
    'LogisticRegression':LogisticRegression(class_weight='balanced',
                                            max_iter=1000,
                                            random_state=42),
    "KNeighborsClassifier":KNeighborsClassifier(),
    "SVC":SVC(class_weight='balanced',probability=True,random_state=42)
}
model_list=[]
ac_list=[]
train_ac_list=[]
train_f1_list = []
test_f1_list = []
train_recall=[]
test_recall=[]

for i in range(len(models)):
    model=list(models.values())[i]
    model.fit(X_train,y_train)
    
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)
    
    y_train_acc,y_train_f1,y_train_precision,y_train_recall=evaluate(y_train,y_train_pred)
    y_test_acc,y_test_f1,y_test_precision,y_test_recall=evaluate(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    ac_list.append(y_test_acc)
    train_ac_list.append(y_train_acc)
    train_f1_list.append(y_train_f1)
    test_f1_list.append(y_test_f1)
    train_recall.append(y_train_recall)
    test_recall.append(y_test_recall)
    
    print('-------------------------------')
    
    print("Train performance")
    print('accracy score:{:.4f}'.format(y_train_acc))
    print('f1 score:{:.4f}'.format(y_train_f1))
    print('precision score :{:.4f}'.format(y_train_precision))
    print('recall score :{:.4f}'.format(y_train_recall))
    
    print('\n')
    
    print("Test performance")
    print('accracy score:{:.4f}'.format(y_test_acc))
    print('f1 score:{:.4f}'.format(y_test_f1))
    print('precision score :{:.4f}'.format(y_test_precision))
    print('recall score :{:.4f}'.format(y_test_recall))

RandomForestClassifier
-------------------------------
Train performance
accracy score:0.9987
f1 score:0.9975
precision score :0.9971
recall score :0.9979


Test performance
accracy score:0.7895
f1 score:0.5466
precision score :0.6390
recall score :0.4775
GradientBoostingClassifier
-------------------------------
Train performance
accracy score:0.8265
f1 score:0.6339
precision score :0.7220
recall score :0.5649


Test performance
accracy score:0.7992
f1 score:0.5871
precision score :0.6469
recall score :0.5375
DecisionTreeClassifier
-------------------------------
Train performance
accracy score:0.9983
f1 score:0.9968
precision score :0.9943
recall score :0.9993


Test performance
accracy score:0.7321
f1 score:0.4897
precision score :0.4956
recall score :0.4839
LogisticRegression
-------------------------------
Train performance
accracy score:0.7543
f1 score:0.6364
precision score :0.5245
recall score :0.8088


Test performance
accracy score:0.7332
f1 score:0.6134
precision score :0.49

In [78]:
pd.DataFrame(
    list(zip(
        model_list,
        ac_list,
        train_ac_list,
        train_f1_list,
        test_f1_list,
        train_recall,
        test_recall
    )),
    columns=[
        'models',
        'testing_accuracy',
        'train_accuracy',
        'train_f1',
        'test_f1',
        "train_recall",
        "test_recall"
    ]
).sort_values(
    by=["test_recall","test_f1",'testing_accuracy'],
    ascending=False
)

,models,testing_accuracy,train_accuracy,train_f1,test_f1,train_recall,test_recall
3,LogisticRegression,0.733220,0.754266,0.636364,0.613355,0.808845,0.796574
5,SVC,0.726394,0.767539,0.655811,0.605414,0.833096,0.790150
4,KNeighborsClassifier,0.763936,0.834850,0.679426,0.563617,0.658345,0.573876
1,GradientBoostingClassifier,0.799204,0.826507,0.633854,0.587135,0.564907,0.537473
2,DecisionTreeClassifier,0.732082,0.998294,0.996798,0.489707,0.999287,0.483940
0,RandomForestClassifier,0.789534,0.998673,0.997504,0.546569,0.997860,0.477516


In [84]:
lr_params = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "max_iter": [200, 500, 1000],
    "class_weight": [None, "balanced"]
}

In [85]:
randomCV_model=[
    ('LG',LogisticRegression(),lr_params)
]

In [86]:
model_params={}

In [88]:
import warnings
warnings.filterwarnings('ignore')

In [89]:
from sklearn.model_selection import RandomizedSearchCV
for name,model,params in randomCV_model:    
        Rc=RandomizedSearchCV(estimator=model,param_distributions=lr_params,n_iter=50,scoring='f1',
                      cv=3,verbose=2,random_state=10,n_jobs=-1)
        
        Rc.fit(X_train,y_train)
        model_params[name]=Rc.best_params_
        
        for model_name in model_params:
            print(f"_______________best param for {model_name}________")
            print(model_params[model_name])

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call
    result, _ = _get_response_valu

[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.1, class_weight=balanced, max_iter=200, penalty=l1, solver=liblinear; total time=   0.1s


/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 380,

[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.9s
[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.9s
[CV] END C=10, class_weight=None, max_iter=200, penalty=l2, solver=saga; total time=   0.9s


...

[CV] END C=1, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   0.4s
[CV] END C=1, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   0.6s
[CV] END C=10, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   1.9s
[CV] END C=0.001, class_weight=balanced, max_iter=200, penalty=l2, solver=liblinear; total time=   0.0s
[CV] END C=1, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   0.3s
[CV] END C=10, class_weight=balanced, max_iter=200, penalty=l1, solver=saga; total time=   1.9s
[CV] END C=0.001, class_weight=balanced, max_iter=200, penalty=l2, solver=liblinear; total time=   0.1s
[CV] END C=1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.001, class_weight=balanced, max_iter=200, penalty=l2, solver=liblinear; total time=   0.1s
[CV] END C=0.1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.1s
[CV] END C=0.1, c

/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call
    result, _ = _get_response_valu

[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.2s
[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.1s
[CV] END C=0.01, class_weight=None, max_iter=500, penalty=l1, solver=saga; total time=   0.1s
[CV] END C=1, class_weight=None, max_iter=500, penalty=l1, solver=liblinear; total time=   0.8s


/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "/Users/macbookpro/.pyenv/versions/cv-3.9/lib/python3.9/site-packages/sklearn/metrics/_scorer.py", line 380,

[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   2.8s
[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   2.9s
[CV] END C=100, class_weight=None, max_iter=1000, penalty=l2, solver=saga; total time=   2.9s
_______________best param for LG________
{'solver': 'saga', 'penalty': 'l2', 'max_iter': 200, 'class_weight': None, 'C': 10}


In [93]:
models = {
    "Logistic Regression": LogisticRegression(
        solver='saga',
        penalty='l2',
        max_iter=200,
        class_weight='balanced',
        C=10
    )
}

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_acc, y_train_f1, y_train_precision, y_train_recall = evaluate(
        y_train, y_train_pred
    )

    y_test_acc, y_test_f1, y_test_precision, y_test_recall = evaluate(
        y_test, y_test_pred
    )

    print('-------------------------------')

    print("Train performance")
    print('accuracy score:{:.8f}'.format(y_train_acc))
    print('f1 score:{:.8f}'.format(y_train_f1))
    print('precision score:{:.8f}'.format(y_train_precision))
    print('recall score:{:.8f}'.format(y_train_recall))

    print('\n')

    print("Test performance")
    print('accuracy score:{:.8f}'.format(y_test_acc))
    print('f1 score:{:.8f}'.format(y_test_f1))
    print('precision score:{:.8f}'.format(y_test_precision))
    print('recall score:{:.8f}'.format(y_test_recall))

-------------------------------
Train performance
accuracy score:0.75255973
f1 score:0.63496503
precision score:0.52231937
recall score:0.80955777


Test performance
accuracy score:0.73265074
f1 score:0.61221122
precision score:0.49798658
recall score:0.79443255
